<a href="https://colab.research.google.com/github/Shaimaa307/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shaimaa307/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice and why

I use a Decision Tree classifier because it is simple, easy to interpret, and suitable as a baseline machine learning model.

The model predicts whether a content page needs action using search performance signals. It will be compared with the Week-4 baseline using the same data and evaluation metric.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb huggingface_hub scikit-learn

from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

warehouse = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT *
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 100000
"""

df = con.sql(query).df()

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(100000, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split design

The dataset is randomly split into 80% training data and 20% testing data.

The same dataset is used for both the baseline and the Decision Tree model, making the comparison fair. The test set is not used during training.

In [15]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [23]:
from sklearn.model_selection import train_test_split
import numpy as np

# Fill missing values
cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "sessions_organic",
    "scroll_events"
]

for c in cols:
    df[c] = df[c].fillna(0)

# Calculate CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

# Target
df["target"] = (df["ctr"] < 0.05).astype(int)

# Features (without leakage)
X = df[
    [
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "sessions_organic",
        "scroll_events"
    ]
]

y = df["target"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Train + compare vs my baseline

The Decision Tree model is trained using search performance features.

The model is compared with the Week-4 baseline using the same train/test split and the same evaluation metric (Accuracy).

In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Train model
model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
pred = model.predict(X_test)

model_acc = accuracy_score(
    y_test,
    pred
)

# -----------------------------
# Week-4 Baseline
# -----------------------------

# نفس الـ split المستخدم للموديل
_, baseline_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

baseline_pred = (
    baseline_test["gsc_clicks"] == 0
).astype(int)

baseline_acc = accuracy_score(
    y_test,
    baseline_pred
)

# Comparison table
comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Decision Tree"
    ],
    "Accuracy": [
        baseline_acc,
        model_acc
    ]
})

comparison

,Method,Accuracy
0,Week-4 Baseline,0.95005
1,Decision Tree,0.99630


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and interpretation

The Decision Tree achieved higher overall accuracy than the Week-4 baseline.

However, the dataset is highly imbalanced, with most pages belonging to one class. The model performs well on the majority class but fails to identify the minority class.

The most influential feature is `gsc_avg_position`, followed by `sessions_organic` and `ga4_pageviews`.

These results should be interpreted as decision support rather than evidence of causation.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import classification_report
import pandas as pd

print(classification_report(y_test, pred))

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        73
           1       1.00      1.00      1.00     19927

    accuracy                           1.00     20000
   macro avg       0.50      0.50      0.50     20000
weighted avg       0.99      1.00      0.99     20000



,Feature,Importance
0,gsc_avg_position,0.607911
3,sessions_organic,0.267556
1,ga4_pageviews,0.102928
2,ga4_sessions,0.021606
4,scroll_events,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.